# 03 — Hyperparameter Tuning
## SkillAlign AI — Optimasi Model
**Tim:** CC26-PSU318 | **Capstone Project** Coding Camp 2026

---
Notebook ini melakukan hyperparameter tuning menggunakan Keras Tuner
untuk menemukan konfigurasi model optimal.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import tensorflow as tf
import keras_tuner as kt
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from src.preprocessing.nlp_preprocessor import NLPPreprocessor
from src.training.hyperparameter_tuning import HyperparameterTuner
from src.models.custom_loss import focal_loss
from src.utils.metrics import compute_all_metrics, compute_classification_report, check_performance_targets
from src.utils.visualization import TrainingVisualizer

print(f'TensorFlow: {tf.__version__}')
print(f'Keras Tuner: {kt.__version__}')
print('Setup complete!')

TensorFlow: 2.21.0
Keras Tuner: 1.4.8
Setup complete!


## 2. Load Preprocessed Data
Gunakan preprocessor yang sudah di-save dari notebook 02.

In [2]:
# Cek apakah preprocessor sudah ada dari notebook 02
preprocessor_path = '../preprocessors/nlp_preprocessor.pkl'
if os.path.exists(preprocessor_path):
    preprocessor = joblib.load(preprocessor_path)
    print(f'Preprocessor loaded. Vocab size: {preprocessor.vocab_size}')
    print('Lanjut ke section 2b untuk load data yang sudah ada.')
else:
    print('Preprocessor belum ada. Jalankan notebook 02 terlebih dahulu,')
    print('atau lanjut ke section 2a untuk prepare data dari awal.')

Preprocessor loaded. Vocab size: 10000
Lanjut ke section 2b untuk load data yang sudah ada.


In [3]:
# ============================================
# 2a. Prepare data jika belum ada preprocessor
# (Skip jika sudah load dari notebook 02)
# ============================================
if not os.path.exists(preprocessor_path):
    # Load dan prepare data (sama seperti notebook 02)
    df_posting = pd.read_csv(
        '../Dataset/database_design/job_posting.csv',
        usecols=['job_posting_id', 'title', 'job_description', 'skills_desc']
    ).dropna(subset=['job_description'])
    
    df_job_skills = pd.read_csv('../Dataset/database_design/job_skills.csv')
    df_skill_names = pd.read_csv('../Dataset/database_design/skills.csv')
    df_job_skills = df_job_skills.merge(df_skill_names, on='skill_id', how='left')
    job_skills_grouped = df_job_skills.groupby('job_posting_id')['skill_name'].apply(list).reset_index()
    job_skills_grouped.columns = ['job_posting_id', 'required_skills']
    
    df_pairs = df_posting.merge(job_skills_grouped, on='job_posting_id', how='inner')
    df_pairs = df_pairs.dropna(subset=['job_description', 'required_skills'])
    
    all_skills = df_skill_names['skill_name'].dropna().unique().tolist()
    np.random.seed(42)
    
    cv_texts, job_texts, labels = [], [], []
    sample_size = min(len(df_pairs), 15000)  # Sedikit lebih kecil untuk tuning
    df_sample = df_pairs.sample(n=sample_size, random_state=42).reset_index(drop=True)
    
    for _, row in df_sample.iterrows():
        job_desc = str(row['job_description'])[:2000]
        req_skills = row['required_skills'] if isinstance(row['required_skills'], list) else []
        title = str(row.get('title', 'Professional'))
        
        # Positive pair
        n = max(1, int(len(req_skills) * np.random.uniform(0.6, 1.0)))
        matched = np.random.choice(req_skills, min(n, len(req_skills)), replace=False).tolist()
        cv_pos = f"Experienced {title} with skills in {', '.join(matched)}. {np.random.randint(1,10)} years exp."
        cv_texts.append(cv_pos); job_texts.append(job_desc); labels.append(1)
        
        # Negative pair
        other = [s for s in all_skills if s not in req_skills]
        rand_s = np.random.choice(other if len(other)>=3 else all_skills, min(5,max(len(other),3)), replace=False).tolist()
        cv_neg = f"Professional with expertise in {', '.join(rand_s)}. {np.random.randint(1,8)} years exp."
        cv_texts.append(cv_neg); job_texts.append(job_desc); labels.append(0)
    
    labels = np.array(labels, dtype=np.float32)
    
    MAX_VOCAB_SIZE = 10000
    MAX_SEQ_LEN = 300
    preprocessor = NLPPreprocessor(max_vocab_size=MAX_VOCAB_SIZE, max_seq_len=MAX_SEQ_LEN)
    all_t = cv_texts + job_texts
    preprocessor.fit(all_t)
    cv_sequences = preprocessor.transform(cv_texts)
    job_sequences = preprocessor.transform(job_texts)
    print(f'Data prepared. Pairs: {len(labels):,}, Vocab: {preprocessor.vocab_size}')
else:
    # Jika preprocessor sudah ada, re-generate data dengan preprocessor yang sama
    # (Idealnya data sequences juga di-save, tapi untuk tuning kita regenerate)
    df_posting = pd.read_csv(
        '../Dataset/database_design/job_posting.csv',
        usecols=['job_posting_id', 'title', 'job_description', 'skills_desc']
    ).dropna(subset=['job_description'])
    
    df_job_skills = pd.read_csv('../Dataset/database_design/job_skills.csv')
    df_skill_names = pd.read_csv('../Dataset/database_design/skills.csv')
    df_job_skills = df_job_skills.merge(df_skill_names, on='skill_id', how='left')
    job_skills_grouped = df_job_skills.groupby('job_posting_id')['skill_name'].apply(list).reset_index()
    job_skills_grouped.columns = ['job_posting_id', 'required_skills']
    
    df_pairs = df_posting.merge(job_skills_grouped, on='job_posting_id', how='inner')
    df_pairs = df_pairs.dropna(subset=['job_description', 'required_skills'])
    all_skills = df_skill_names['skill_name'].dropna().unique().tolist()
    np.random.seed(42)
    
    cv_texts, job_texts, labels = [], [], []
    sample_size = min(len(df_pairs), 15000)
    df_sample = df_pairs.sample(n=sample_size, random_state=42).reset_index(drop=True)
    
    for _, row in df_sample.iterrows():
        job_desc = str(row['job_description'])[:2000]
        req_skills = row['required_skills'] if isinstance(row['required_skills'], list) else []
        title = str(row.get('title', 'Professional'))
        n = max(1, int(len(req_skills) * np.random.uniform(0.6, 1.0)))
        matched = np.random.choice(req_skills, min(n, len(req_skills)), replace=False).tolist()
        cv_pos = f"Experienced {title} with skills in {', '.join(matched)}. {np.random.randint(1,10)} years exp."
        cv_texts.append(cv_pos); job_texts.append(job_desc); labels.append(1)
        other = [s for s in all_skills if s not in req_skills]
        rand_s = np.random.choice(other if len(other)>=3 else all_skills, min(5,max(len(other),3)), replace=False).tolist()
        cv_neg = f"Professional with expertise in {', '.join(rand_s)}. {np.random.randint(1,8)} years exp."
        cv_texts.append(cv_neg); job_texts.append(job_desc); labels.append(0)
    
    labels = np.array(labels, dtype=np.float32)
    MAX_SEQ_LEN = preprocessor.max_seq_len
    cv_sequences = preprocessor.transform(cv_texts)
    job_sequences = preprocessor.transform(job_texts)
    print(f'Data prepared with existing preprocessor. Pairs: {len(labels):,}')

Data prepared with existing preprocessor. Pairs: 30,000


In [4]:
# Train-Test-Val split
cv_train, cv_test, job_train, job_test, y_train, y_test = train_test_split(
    cv_sequences, job_sequences, labels, test_size=0.2, random_state=42, stratify=labels
)
cv_train, cv_val, job_train, job_val, y_train, y_val = train_test_split(
    cv_train, job_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)
print(f'Train: {len(y_train):,} | Val: {len(y_val):,} | Test: {len(y_test):,}')

Train: 20,400 | Val: 3,600 | Test: 6,000


## 3. Hyperparameter Tuning

In [5]:
# Setup tuner
tuner = HyperparameterTuner(
    vocab_size=preprocessor.vocab_size,
    max_seq_len=MAX_SEQ_LEN,
    embedding_dim=128,
    project_name='skillalign_tuning',
    directory='../logs/tuning'
)
print('Tuner initialized.')

Tuner initialized.


In [6]:
# Run search
# Sesuaikan max_trials dan epochs_per_trial berdasarkan resource
best_model = tuner.search(
    x_train=[cv_train, job_train],
    y_train=y_train,
    x_val=[cv_val, job_val],
    y_val=y_val,
    max_trials=10,        # Tambah jika resource cukup
    epochs_per_trial=15,
    batch_size=32
)
print('\nHyperparameter search complete!')

Trial 10 Complete [00h 04m 25s]
val_accuracy: 1.0

Best val_accuracy So Far: 1.0
Total elapsed time: 01h 24m 06s

Hyperparameter search complete!


In [7]:
# Best hyperparameters
best_hp = tuner.get_best_hyperparameters()
print('\n=== Best Hyperparameters ===')
for k, v in best_hp.items():
    print(f'  {k}: {v}')


=== Best Hyperparameters ===
  attention_units: 192
  conv_filters_1: 256
  conv_filters_2: 128
  dense_units_1: 128
  dense_units_2: 256
  dropout_rate: 0.2
  learning_rate: 0.0005988959230738318
  kernel_size: 3


In [8]:
# Search summary
tuner.print_search_summary()

Results summary
Results in ../logs/tuning\skillalign_tuning
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 00 summary
Hyperparameters:
attention_units: 192
conv_filters_1: 256
conv_filters_2: 128
dense_units_1: 128
dense_units_2: 256
dropout_rate: 0.2
learning_rate: 0.0005988959230738318
kernel_size: 3
Score: 1.0

Trial 01 summary
Hyperparameters:
attention_units: 128
conv_filters_1: 192
conv_filters_2: 96
dense_units_1: 384
dense_units_2: 192
dropout_rate: 0.4
learning_rate: 0.00011451155720572537
kernel_size: 5
Score: 1.0

Trial 02 summary
Hyperparameters:
attention_units: 192
conv_filters_1: 64
conv_filters_2: 64
dense_units_1: 128
dense_units_2: 192
dropout_rate: 0.30000000000000004
learning_rate: 0.0018578460759471264
kernel_size: 7
Score: 1.0

Trial 03 summary
Hyperparameters:
attention_units: 192
conv_filters_1: 256
conv_filters_2: 128
dense_units_1: 512
dense_units_2: 128
dropout_rate: 0.4
learning_rate: 0.0006179579394890727
kernel_size: 3
Score:

## 4. Evaluate Best Model

In [9]:
# Evaluate best model on test set
test_results = best_model.evaluate([cv_test, job_test], y_test, verbose=1)
print(f'\nTest results: {dict(zip(best_model.metrics_names, test_results))}')

188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 1.0000 - loss: 0.0030

Test results: {'loss': 0.0030338605865836143, 'compile_metrics': 1.0}


In [10]:
# Detailed metrics
y_pred = best_model.predict([cv_test, job_test], verbose=0)
metrics = compute_all_metrics(y_test, y_pred)
print('\n=== Metrics (Best Model) ===')
for k, v in metrics.items():
    print(f'  {k}: {v}')

print('\n=== Classification Report ===')
print(compute_classification_report(y_test, y_pred))

# Performance targets
targets = check_performance_targets(metrics)
print('\n=== Performance Targets ===')
for name, info in targets.items():
    status = 'PASS' if info['passed'] else 'FAIL'
    print(f"  {name}: {info['value']} (target: {info['target']}) [{status}]")


=== Metrics (Best Model) ===
  accuracy: 1.0
  precision: 1.0
  recall: 1.0
  f1_score: 1.0
  mae: 0.098
  auc: 1.0

=== Classification Report ===
              precision    recall  f1-score   support

   Not Match       1.00      1.00      1.00      3000
       Match       1.00      1.00      1.00      3000

    accuracy                           1.00      6000
   macro avg       1.00      1.00      1.00      6000
weighted avg       1.00      1.00      1.00      6000


=== Performance Targets ===
  accuracy: 1.0 (target: >= 0.85) [PASS]
  mae: 0.098 (target: <= 0.02) [FAIL]


In [11]:
# Confusion matrix & plots
viz = TrainingVisualizer(save_dir='../logs/plots')
y_pred_binary = (y_pred > 0.5).astype(int).flatten()
viz.plot_confusion_matrix(y_test, y_pred_binary, filename='confusion_matrix_tuned.png')
viz.plot_score_distribution(y_pred.flatten(), y_test, filename='score_dist_tuned.png')
viz.plot_metrics_comparison(
    metrics, targets={'accuracy': 0.85, 'f1_score': 0.80},
    filename='metrics_tuned.png'
)
print('Plots saved.')

Plots saved.


## 5. Retrain Best Model (Full Epochs)

In [12]:
# Retrain best model with more epochs dan full callbacks
from src.training.train import ModelTrainer

trainer = ModelTrainer(
    model=best_model,
    config={
        'batch_size': 32,
        'epochs': 50,
        'learning_rate': best_hp.get('learning_rate', 0.001),
        'early_stopping_patience': 10,
        'reduce_lr_patience': 5,
        'f1_patience': 8,
        'f1_threshold': 0.80
    },
    log_dir='../logs/training_tuned',
    model_dir='../models'
)

# Recompile is not needed since best_model is already compiled
history = trainer.train(
    x_train=[cv_train, job_train],
    y_train=y_train,
    x_val=[cv_val, job_val],
    y_val=y_val
)
print('Retrain complete!')

Epoch 1/50
638/638 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 1.0000 - loss: 7.5670e-07
Epoch 1: val_accuracy improved from None to 1.00000, saving model to ../models\best_model.keras

  [F1Callback] Epoch 1: F1=1.0000 | Precision=1.0000 | Recall=1.0000 | Best F1=0.0000
  [F1Callback] Model saved to ../models\best_f1_model.keras (F1=1.0000)

  [F1Callback] F1-score 1.0000 >= threshold 0.8000. Target tercapai!
638/638 ━━━━━━━━━━━━━━━━━━━━ 65s 98ms/step - accuracy: 1.0000 - loss: 1.9737e-07 - val_accuracy: 1.0000 - val_loss: 2.0981e-13 - learning_rate: 5.9890e-04 - val_f1_score: 1.0000 - val_precision_custom: 1.0000 - val_recall_custom: 1.0000
Epoch 2/50
637/638 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 1.0000 - loss: 6.1909e-08
Epoch 2: val_accuracy did not improve from 1.00000

  [F1Callback] Epoch 2: F1=1.0000 | Precision=1.0000 | Recall=1.0000 | Best F1=1.0000
  [F1Callback] No improvement. Patience: 1/8
638/638 ━━━━━━━━━━━━━━━━━━━━ 54s 85ms/step - accuracy: 1.0000 - loss: 4.3

In [13]:
# Plot retrained history
viz.plot_training_history(history, filename='training_history_tuned.png')

# Final evaluation
y_pred_final = best_model.predict([cv_test, job_test], verbose=0)
final_metrics = compute_all_metrics(y_test, y_pred_final)
print('\n=== Final Metrics (After Retrain) ===')
for k, v in final_metrics.items():
    print(f'  {k}: {v}')


=== Final Metrics (After Retrain) ===
  accuracy: 1.0
  precision: 1.0
  recall: 1.0
  f1_score: 1.0
  mae: 0.0
  auc: 1.0


## 6. Save Tuned Model

In [14]:
# Save tuned model
tuned_path = trainer.save_model('skillalign_matcher_tuned.keras')
print(f'Tuned model saved: {tuned_path}')

# Save best hyperparameters
import json
hp_path = '../models/best_hyperparameters.json'
with open(hp_path, 'w') as f:
    json.dump(best_hp, f, indent=2, default=str)
print(f'Best hyperparameters saved: {hp_path}')

# Save final metrics
metrics_path = '../models/final_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(final_metrics, f, indent=2, default=str)
print(f'Final metrics saved: {metrics_path}')
print('\n=== Done! ===')

Tuned model saved: ../models\skillalign_matcher_tuned.keras
Best hyperparameters saved: ../models/best_hyperparameters.json
Final metrics saved: ../models/final_metrics.json

=== Done! ===


In [2]:
# Test inference
from src.inference.predict import SkillAlignPredictor

predictor = SkillAlignPredictor(
    model_path='../models/skillalign_matcher_tuned.keras',
    preprocessor_path='../preprocessors/nlp_preprocessor.pkl'
)
predictor.load()

# Test case
result = predictor.predict(
    cv_text='Experienced Data Scientist with 5 years in Python, TensorFlow, machine learning, deep learning, and data analysis.',
    job_description='Looking for a Data Scientist with strong Python skills, experience in ML frameworks, TensorFlow, and statistical analysis.'
)

print(f'Matching Score: {result.matching_score}')
print(f'Confidence: {result.confidence}')
print(f'Recommendation: {result.recommendation}')
print(f'Inference Time: {result.inference_time_ms:.1f}ms')


Matching Score: 1.0
Confidence: High
Recommendation: Highly Recommended
Inference Time: 2893.4ms
